[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvvvvivekkk/AI-Engineer/blob/main/10-agentic-ai/notebooks/02-react-loop-from-scratch.ipynb)

# Notebook 02 — A ReAct agent loop from scratch

**Goal:** build the agent loop yourself in plain Python and watch every step.

**ReAct** = **Re**ason + **Act**. The model repeats *Thought → Action → Observation* until it writes *Final Answer*.

So the notebook runs with no internet and gives the same result every time, the "LLM" is a **scripted fake LLM**: an object that hands back pre-written replies one by one. The **loop is real** — swapping in a real model changes one line.

## Step 1 — Tools

Two small tools: a calculator (which refuses anything that is not arithmetic — a first taste of safety) and a tiny search over fake notes.

In [1]:
import re

def calculator(expression: str) -> str:
    if not re.fullmatch(r"[0-9+\-*/(). ]+", expression):
        return "Error: only arithmetic is allowed"
    return str(eval(expression))

KB = {"taipei 101 height": "Taipei 101 is 508 metres tall.",
      "eiffel tower height": "The Eiffel Tower is about 330 metres tall.",
      "burj khalifa height": "The Burj Khalifa is 828 metres tall."}

def search_notes(query: str) -> str:
    for key, value in KB.items():
        if all(w in key for w in query.lower().split()):
            return value
    return "No result found."

TOOLS = {"calculator": calculator, "search_notes": search_notes}
print(calculator("508 - 330"), "|", calculator("__import__('os')"), "|", search_notes("eiffel height"))

178 | Error: only arithmetic is allowed | The Eiffel Tower is about 330 metres tall.


## Step 2 — The prompt format and the fake LLM

The **system prompt** teaches the model the format. A real model reads the whole **transcript** (everything so far) and writes the next step. Our `ScriptedLLM` ignores the transcript and returns the next scripted line — but it has the *same interface*: text in, text out.

In [2]:
SYSTEM_PROMPT = """You are a helpful agent. Use this format:
Thought: <reasoning>
Action: <tool>[<input>]
You will then get: Observation: <result>
When done: Final Answer: <answer>
Tools: calculator[expression], search_notes[query]"""

class ScriptedLLM:
    def __init__(self, replies):
        self.replies = list(replies)
        self.calls = 0
    def __call__(self, transcript: str) -> str:
        self.calls += 1
        return self.replies.pop(0) if self.replies else "Final Answer: I am out of ideas."

print(SYSTEM_PROMPT)

You are a helpful agent. Use this format:
Thought: <reasoning>
Action: <tool>[<input>]
You will then get: Observation: <result>
When done: Final Answer: <answer>
Tools: calculator[expression], search_notes[query]


## Step 3 — The loop

Read this cell slowly; it is the whole idea of an agent:

1. ask the LLM for the next step,
2. stop if it gave a Final Answer,
3. otherwise find the `Action: tool[input]`,
4. run the tool,
5. append `Observation: ...` and go round again — but never more than `max_steps` times.

In [3]:
ACTION_RE = re.compile(r"Action:\s*(\w+)\[(.*)\]")

def run_agent(question, llm, max_steps=6, verbose=True):
    transcript = f"{SYSTEM_PROMPT}\n\nQuestion: {question}\n"
    for step in range(1, max_steps + 1):
        reply = llm(transcript)
        transcript += reply + "\n"
        if verbose:
            print(f"--- step {step} ---\n{reply}")
        if "Final Answer:" in reply:
            return reply.split("Final Answer:", 1)[1].strip()
        m = ACTION_RE.search(reply)
        if not m:
            obs = "Error: no Action found. Use Action: tool[input]."
        else:
            name, arg = m.group(1), m.group(2)
            obs = TOOLS[name](arg) if name in TOOLS else f"Error: unknown tool '{name}'. Use one of {list(TOOLS)}"
        transcript += f"Observation: {obs}\n"
        if verbose:
            print(f"Observation: {obs}")
    return "Stopped: max_steps reached."
print("loop defined")

loop defined


## Step 4 — Run it

The question needs **two lookups and one calculation**. Watch the Thought → Action → Observation rhythm.

In [4]:
script = [
    "Thought: I need Taipei 101's height.\nAction: search_notes[taipei 101 height]",
    "Thought: 508 m. Now the Eiffel Tower.\nAction: search_notes[eiffel tower height]",
    "Thought: Subtract.\nAction: calculator[508 - 330]",
    "Thought: 178 m.\nFinal Answer: Taipei 101 is about 178 metres taller than the Eiffel Tower.",
]
answer = run_agent("How much taller is Taipei 101 than the Eiffel Tower?", ScriptedLLM(script))
print("\nANSWER:", answer)

--- step 1 ---
Thought: I need Taipei 101's height.
Action: search_notes[taipei 101 height]
Observation: Taipei 101 is 508 metres tall.
--- step 2 ---
Thought: 508 m. Now the Eiffel Tower.
Action: search_notes[eiffel tower height]
Observation: The Eiffel Tower is about 330 metres tall.
--- step 3 ---
Thought: Subtract.
Action: calculator[508 - 330]
Observation: 178
--- step 4 ---
Thought: 178 m.
Final Answer: Taipei 101 is about 178 metres taller than the Eiffel Tower.

ANSWER: Taipei 101 is about 178 metres taller than the Eiffel Tower.


## Step 5 — What goes wrong (and how the loop protects us)

Real models make mistakes. Here the fake LLM (a) asks for a tool that does not exist, (b) forgets the format, then (c) recovers. The loop returns **error observations** instead of crashing, so the model can correct itself.

In [5]:
messy = [
    "Thought: I'll use the internet.\nAction: google[burj khalifa]",
    "I think it's tall.",
    "Thought: Use the right tool.\nAction: search_notes[burj khalifa height]",
    "Final Answer: The Burj Khalifa is 828 metres tall.",
]
print("ANSWER:", run_agent("How tall is the Burj Khalifa?", ScriptedLLM(messy)))

--- step 1 ---
Thought: I'll use the internet.
Action: google[burj khalifa]
Observation: Error: unknown tool 'google'. Use one of ['calculator', 'search_notes']
--- step 2 ---
I think it's tall.
Observation: Error: no Action found. Use Action: tool[input].
--- step 3 ---
Thought: Use the right tool.
Action: search_notes[burj khalifa height]
Observation: The Burj Khalifa is 828 metres tall.
--- step 4 ---
Final Answer: The Burj Khalifa is 828 metres tall.
ANSWER: The Burj Khalifa is 828 metres tall.


And if the model never finishes (it keeps calling the same tool), `max_steps` stops the loop. Without this limit an agent can burn money forever.

In [6]:
stuck = ["Thought: check again.\nAction: calculator[1+1]"] * 10
print("ANSWER:", run_agent("What is 1+1?", ScriptedLLM(stuck), max_steps=3))

--- step 1 ---
Thought: check again.
Action: calculator[1+1]
Observation: 2
--- step 2 ---
Thought: check again.
Action: calculator[1+1]
Observation: 2
--- step 3 ---
Thought: check again.
Action: calculator[1+1]
Observation: 2
ANSWER: Stopped: max_steps reached.


## Step 6 — The same loop with native tool calls

Modern APIs return **structured tool calls** instead of text to parse (see notebook 01). The loop is the same — only the parsing step disappears. Here the fake LLM returns dicts shaped like an Anthropic reply. The `messages` list is the transcript.

In [7]:
import json

class ScriptedToolLLM:
    """Returns Anthropic-shaped replies: a list of content blocks + a stop_reason."""
    def __init__(self, replies): self.replies = list(replies)
    def __call__(self, messages, tools): return self.replies.pop(0)

def native_agent(question, llm, max_steps=5):
    messages = [{"role": "user", "content": question}]
    for step in range(max_steps):
        reply = llm(messages, tools=list(TOOLS))
        messages.append({"role": "assistant", "content": reply["content"]})
        if reply["stop_reason"] != "tool_use":
            return reply["content"][0]["text"], messages
        results = []
        for b in reply["content"]:
            if b["type"] == "tool_use":
                out = TOOLS[b["name"]](**b["input"])
                print(f"step {step+1}: {b['name']}({b['input']}) -> {out}")
                results.append({"type": "tool_result", "tool_use_id": b["id"], "content": out})
        messages.append({"role": "user", "content": results})
    return "Stopped", messages

fake = ScriptedToolLLM([
    {"stop_reason": "tool_use", "content": [{"type": "tool_use", "id": "t1", "name": "search_notes", "input": {"query": "burj khalifa height"}},
                                             {"type": "tool_use", "id": "t2", "name": "search_notes", "input": {"query": "taipei 101 height"}}]},
    {"stop_reason": "tool_use", "content": [{"type": "tool_use", "id": "t3", "name": "calculator", "input": {"expression": "828 / 508"}}]},
    {"stop_reason": "end_turn", "content": [{"type": "text", "text": "The Burj Khalifa is about 1.63 times as tall as Taipei 101."}]},
])
ans, history = native_agent("How many times taller is the Burj Khalifa than Taipei 101?", fake)
print("\nANSWER:", ans)
print("messages in history:", len(history))

step 1: search_notes({'query': 'burj khalifa height'}) -> The Burj Khalifa is 828 metres tall.
step 1: search_notes({'query': 'taipei 101 height'}) -> Taipei 101 is 508 metres tall.
step 2: calculator({'expression': '828 / 508'}) -> 1.6299212598425197

ANSWER: The Burj Khalifa is about 1.63 times as tall as Taipei 101.
messages in history: 6


Notice the model asked for **two tool calls in one turn** (parallel tool calls). Your loop must run all of them and send all results back together.

## Step 7 — Plug in a real model (optional)

`real_llm` below has the same shape as `ScriptedToolLLM`. With a key, the same `native_agent` drives Claude. Without one, it is skipped.

In [8]:
import os

def make_real_llm():
    import anthropic
    client = anthropic.Anthropic()
    schemas = [
        {"name": "calculator", "description": "Evaluate arithmetic.", "input_schema": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]}},
        {"name": "search_notes", "description": "Look up facts about building heights.", "input_schema": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}},
    ]
    def llm(messages, tools):
        r = client.messages.create(model=os.getenv("ANTHROPIC_MODEL", "claude-opus-5-5"), max_tokens=500, tools=schemas, messages=messages)
        return {"stop_reason": r.stop_reason, "content": [b.model_dump() for b in r.content]}
    return llm

if os.getenv("ANTHROPIC_API_KEY"):
    print(native_agent("How much taller is the Burj Khalifa than the Eiffel Tower?", make_real_llm())[0])
else:
    print("No ANTHROPIC_API_KEY - real model skipped. The scripted run above used the identical loop.")

No ANTHROPIC_API_KEY - real model skipped. The scripted run above used the identical loop.


## Recap

* An agent is a **loop**: ask → (answer? stop) → run tool → feed back.
* Always have **max_steps**, **unknown-tool errors** and **argument validation**.
* Text ReAct and native tool calling share the same loop; native calling is more reliable.

**Try it:** add a `max_same_call` rule that stops the agent if it makes the exact same tool call twice in a row.